In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
DATA_DIR   = 'dataset'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
# ── Style ─────────────────────────────────────────────────────────
BG      = '#0D0F14'
PANEL   = '#13161E'
ACCENT1 = '#00D4FF'
ACCENT2 = '#FF6B35'
ACCENT3 = '#7B61FF'
ACCENT4 = '#39FF14'
MUTED   = '#3A4258'
TEXT    = '#E2E8F0'
SUBTEXT = '#718096'
GROUP_COLORS = {1: ACCENT1, 2: ACCENT2, 3: ACCENT3, 4: ACCENT4}
 
plt.rcParams.update({
    'figure.facecolor':  BG,
    'axes.facecolor':    PANEL,
    'axes.edgecolor':    MUTED,
    'axes.labelcolor':   TEXT,
    'axes.titlecolor':   TEXT,
    'xtick.color':       SUBTEXT,
    'ytick.color':       SUBTEXT,
    'text.color':        TEXT,
    'grid.color':        MUTED,
    'grid.alpha':        0.3,
    'font.family':       'monospace',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        130,
})
 
def save(fig, name):
    path = os.path.join(OUTPUT_DIR, f'{name}.png')
    fig.savefig(path, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f'  ✓  {path}')

In [6]:
# LOAD
print('\nLoading data...')
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train.csv'), index_col='ROW_ID')
y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train.csv'), index_col='ROW_ID')
X_test  = pd.read_csv(os.path.join(DATA_DIR, 'X_test.csv'),  index_col='ROW_ID')
 
train = X_train.copy()
train['TARGET'] = y_train.iloc[:, 0]
train['LABEL']  = (train['TARGET'] > 0).astype(int)
 
ret_cols = [f'RET_{i}'           for i in range(1, 21)]
vol_cols = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]
 
print(f'  Train : {train.shape}')
print(f'  Test  : {X_test.shape}')


Loading data...
  Train : (527073, 46)
  Test  : (31870, 44)


In [16]:
# ═══════════════════════════════════════════════════════════════════
# CLEANUP
# ═══════════════════════════════════════════════════════════════════
print('\nCleaning...')
 
# Drop SIGNED_VOLUME_1 — 73% missing, unusable
for df in [train, X_train, X_test]:
    df.drop(columns=['SIGNED_VOLUME_1'], inplace=True, errors='ignore')
 
vol_cols_clean = [f'SIGNED_VOLUME_{i}' for i in range(2, 21)]
 
# Fill boundary NaNs in RET and SIGNED_VOLUME with 0
# (these are edge effects from the 20-day window at series start, not true gaps)
for df in [train, X_train, X_test]:
    sv_present = [c for c in vol_cols_clean if c in df.columns]
    df[ret_cols]  = df[ret_cols].fillna(0)
    df[sv_present] = df[sv_present].fillna(0)
 
mdt_median = train['MEDIAN_DAILY_TURNOVER'].median()
for df in [train, X_train, X_test]:
    df['MEDIAN_DAILY_TURNOVER'] = df['MEDIAN_DAILY_TURNOVER'].fillna(mdt_median)
 
# Derived columns used in plots
train['ret_vol']  = train[ret_cols].std(axis=1)
train['ret_mean'] = train[ret_cols].mean(axis=1)
 
print(f'  Remaining missing: {train.isnull().sum().sum()}')


Cleaning...
  Remaining missing: 0


In [17]:
# ═══════════════════════════════════════════════════════════════════
# FIG 1 — DATASET OVERVIEW
# ═══════════════════════════════════════════════════════════════════
print('\n[1/6] Dataset overview...')
fig = plt.figure(figsize=(18, 9), facecolor=BG)
fig.suptitle('FIG 1  ·  DATASET OVERVIEW',
             fontsize=13, fontweight='bold', color=ACCENT1, y=0.99)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.55, wspace=0.45)
 
# 1a. Train vs test size
ax = fig.add_subplot(gs[0, 0])
sizes  = [len(train), len(X_test)]
labels = ['TRAIN', 'TEST']
bars = ax.bar(labels, sizes, color=[ACCENT1, ACCENT2], width=0.45, edgecolor='none')
ax.set_title('DATASET SIZE', fontsize=9, color=SUBTEXT, pad=6)
ax.set_ylabel('Observations', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax.grid(axis='y', alpha=0.2)
for bar, val in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val:,}', ha='center', va='bottom', fontsize=7, color=TEXT)
 
# 1b. Group distribution (train)
ax = fig.add_subplot(gs[0, 1])
gc = train['GROUP'].value_counts().sort_index()
colors = [GROUP_COLORS.get(g, MUTED) for g in gc.index]
bars = ax.bar([f'G{g}' for g in gc.index], gc.values, color=colors, edgecolor='none')
ax.set_title('GROUP DISTRIBUTION (TRAIN)', fontsize=9, color=SUBTEXT, pad=6)
ax.set_ylabel('Count', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))
ax.grid(axis='y', alpha=0.2)
for bar, val in zip(bars, gc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val/1e3:.0f}k', ha='center', va='bottom', fontsize=7, color=SUBTEXT)
 
# 1c. Class balance overall
ax = fig.add_subplot(gs[0, 2])
cb = train['LABEL'].value_counts().sort_index()
wedges, texts, autotexts = ax.pie(
    cb.values, labels=['NEG (0)', 'POS (1)'],
    colors=[ACCENT2, ACCENT1],
    autopct='%1.1f%%',
    textprops={'fontsize': 8, 'color': TEXT},
    wedgeprops={'edgecolor': BG, 'linewidth': 2},
    startangle=90)
for at in autotexts:
    at.set_color(BG); at.set_fontweight('bold')
ax.set_title('CLASS BALANCE', fontsize=9, color=SUBTEXT, pad=6)
 
# 1d. % positive label by group
ax = fig.add_subplot(gs[0, 3])
cb_group = train.groupby('GROUP')['LABEL'].mean()
bars = ax.bar([f'G{g}' for g in cb_group.index],
              cb_group.values * 100,
              color=[GROUP_COLORS.get(g, MUTED) for g in cb_group.index],
              edgecolor='none')
ax.axhline(50, color=TEXT, linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title('% POSITIVE LABEL BY GROUP', fontsize=9, color=SUBTEXT, pad=6)
ax.set_ylabel('% Positive', fontsize=8)
ax.set_ylim(48, 53)
ax.grid(axis='y', alpha=0.2)
for bar, val in zip(bars, cb_group.values):
    ax.text(bar.get_x() + bar.get_width()/2, val*100 + 0.05,
            f'{val*100:.2f}%', ha='center', va='bottom', fontsize=7, color=TEXT)
 
# 1e. Target distribution
ax = fig.add_subplot(gs[1, :2])
clipped = train['TARGET'].clip(
    train['TARGET'].quantile(0.005),
    train['TARGET'].quantile(0.995))
ax.hist(clipped, bins=300, color=ACCENT1, alpha=0.85, edgecolor='none')
ax.axvline(0, color=ACCENT2, linewidth=1.2, linestyle='--', label='zero')
ax.set_title('TARGET DISTRIBUTION  (0.5%–99.5% clipped)', fontsize=9, color=SUBTEXT, pad=6)
ax.set_xlabel('Next-day return', fontsize=8)
ax.set_ylabel('Count', fontsize=8)
ax.grid(axis='y', alpha=0.2)
stats = train['TARGET'].describe()
ax.text(0.98, 0.92,
        f"mean={stats['mean']:.5f}  std={stats['std']:.4f}\n"
        f"min={stats['min']:.4f}    max={stats['max']:.4f}",
        transform=ax.transAxes, ha='right', va='top', fontsize=7, color=SUBTEXT,
        bbox=dict(boxstyle='round,pad=0.3', facecolor=BG, edgecolor=MUTED))
 
# 1f. Target distribution by group
ax = fig.add_subplot(gs[1, 2:])
for g, col in GROUP_COLORS.items():
    sub = train[train['GROUP'] == g]['TARGET'].clip(
        train['TARGET'].quantile(0.005),
        train['TARGET'].quantile(0.995))
    ax.hist(sub, bins=150, alpha=0.5, color=col, label=f'G{g}', edgecolor='none')
ax.axvline(0, color=TEXT, linewidth=1, linestyle='--', alpha=0.6)
ax.set_title('TARGET DISTRIBUTION BY GROUP', fontsize=9, color=SUBTEXT, pad=6)
ax.set_xlabel('Next-day return', fontsize=8)
ax.set_ylabel('Count', fontsize=8)
ax.legend(fontsize=7, framealpha=0.2)
ax.grid(axis='y', alpha=0.2)
 
save(fig, 'fig1_dataset_overview')
 
 



[1/6] Dataset overview...
  ✓  outputs/fig1_dataset_overview.png


In [18]:
# ═══════════════════════════════════════════════════════════════════
# FIG 2 — MISSING VALUES (pre-cleanup, recomputed from raw)
# ═══════════════════════════════════════════════════════════════════
print('[2/6] Missing values...')
 
# Reload raw to show original missingness
X_train_raw = pd.read_csv(os.path.join(DATA_DIR, 'X_train.csv'), index_col='ROW_ID')
X_test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'X_test.csv'),  index_col='ROW_ID')
 
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG)
fig.suptitle('FIG 2  ·  MISSING VALUES  (raw, before cleanup)',
             fontsize=13, fontweight='bold', color=ACCENT1, y=1.01)
 
for ax, (df, title) in zip(axes, [(X_train_raw, 'X_TRAIN'), (X_test_raw, 'X_TEST')]):
    miss_pct = df.isnull().mean() * 100
    miss_pct = miss_pct[miss_pct > 0].sort_values()
    bar_colors = [ACCENT2 if p > 50 else ACCENT3 if p > 5 else ACCENT1
                  for p in miss_pct]
    ax.barh(range(len(miss_pct)), miss_pct.values, color=bar_colors, edgecolor='none')
    ax.set_yticks(range(len(miss_pct)))
    ax.set_yticklabels(
        [c.replace('SIGNED_VOLUME_', 'SV_').replace('RET_', 'R_')
         for c in miss_pct.index], fontsize=7)
    ax.set_xlabel('% Missing', fontsize=8)
    ax.set_title(title, fontsize=10, color=TEXT, pad=8)
    ax.axvline(5,  color=ACCENT3, linestyle=':', linewidth=1, alpha=0.7, label='>5%')
    ax.axvline(50, color=ACCENT2, linestyle=':', linewidth=1, alpha=0.7, label='>50%')
    ax.legend(fontsize=7, framealpha=0.2)
    ax.grid(axis='x', alpha=0.2)
    for i, val in enumerate(miss_pct.values):
        ax.text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=6, color=SUBTEXT)
 
plt.tight_layout()
save(fig, 'fig2_missing_values')
 
 


[2/6] Missing values...
  ✓  outputs/fig2_missing_values.png


In [11]:
# ═══════════════════════════════════════════════════════════════════
# FIG 3 — RETURN SERIES
# ═══════════════════════════════════════════════════════════════════
print('[3/6] Return series...')
fig = plt.figure(figsize=(18, 10), facecolor=BG)
fig.suptitle('FIG 3  ·  RETURN SERIES  (RET_1 = most recent … RET_20 = oldest)',
             fontsize=13, fontweight='bold', color=ACCENT1, y=0.99)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)
 
lags = list(range(1, 21))
 
# 3a. Boxplot per lag
ax = fig.add_subplot(gs[0, :2])
ret_data = [train[c].dropna() for c in ret_cols]
bp = ax.boxplot(ret_data, patch_artist=True, showfliers=False,
                medianprops=dict(color=ACCENT2, linewidth=1.5),
                whiskerprops=dict(color=MUTED),
                capprops=dict(color=MUTED),
                boxprops=dict(linewidth=0.8))
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(ACCENT1)
    patch.set_alpha(0.3 + 0.5 * (i / 20))
    patch.set_edgecolor(MUTED)
ax.set_xticks(range(1, 21))
ax.set_xticklabels([str(i) for i in lags], fontsize=7)
ax.set_xlabel('Lag  (1 = most recent)', fontsize=8)
ax.set_ylabel('Return', fontsize=8)
ax.set_title('RETURN DISTRIBUTION PER LAG  (outliers hidden)', fontsize=9, color=SUBTEXT, pad=6)
ax.axhline(0, color=TEXT, linewidth=0.6, linestyle='--', alpha=0.4)
ax.grid(axis='y', alpha=0.2)
 
# 3b. Mean |return| by lag
ax = fig.add_subplot(gs[0, 2])
mean_abs = [train[c].abs().mean() for c in ret_cols]
ax.plot(lags, mean_abs, color=ACCENT1, linewidth=1.5, marker='o',
        markersize=4, markerfacecolor=ACCENT2)
ax.set_xlabel('Lag', fontsize=8)
ax.set_ylabel('Mean |RET|', fontsize=8)
ax.set_title('MEAN |RETURN| BY LAG', fontsize=9, color=SUBTEXT, pad=6)
ax.grid(alpha=0.2)
 
# 3c. Correlation of each lag with TARGET
ax = fig.add_subplot(gs[1, :2])
corrs = [train[c].corr(train['TARGET']) for c in ret_cols]
bar_colors = [ACCENT1 if c > 0 else ACCENT2 for c in corrs]
ax.bar(lags, corrs, color=bar_colors, edgecolor='none')
ax.axhline(0, color=TEXT, linewidth=0.6, linestyle='--', alpha=0.5)
ax.set_xticks(lags)
ax.set_xticklabels([str(i) for i in lags], fontsize=7)
ax.set_xlabel('Lag  (1 = most recent)', fontsize=8)
ax.set_ylabel('Pearson correlation', fontsize=8)
ax.set_title('CORRELATION: RET_i  ↔  TARGET', fontsize=9, color=SUBTEXT, pad=6)
ax.grid(axis='y', alpha=0.2)
for x, val in zip(lags, corrs):
    ax.text(x, val + (0.001 if val >= 0 else -0.003),
            f'{val:.3f}', ha='center',
            va='bottom' if val >= 0 else 'top',
            fontsize=5.5, color=TEXT, rotation=90)
 
# 3d. RET_1 split by label
ax = fig.add_subplot(gs[1, 2])
for label, col, name in [(1, ACCENT1, 'POS (1)'), (0, ACCENT2, 'NEG (0)')]:
    sub = train[train['LABEL'] == label]['RET_1'].clip(-0.02, 0.02)
    ax.hist(sub, bins=100, alpha=0.6, color=col, label=name,
            edgecolor='none', density=True)
ax.set_xlabel('RET_1', fontsize=8)
ax.set_ylabel('Density', fontsize=8)
ax.set_title('RET_1 SPLIT BY TARGET LABEL', fontsize=9, color=SUBTEXT, pad=6)
ax.legend(fontsize=7, framealpha=0.2)
ax.grid(alpha=0.2)
 
save(fig, 'fig3_return_series')
 
 


[3/6] Return series...
  ✓  outputs/fig3_return_series.png


In [19]:
# ═══════════════════════════════════════════════════════════════════
# FIG 4 — SIGNED VOLUME SERIES
# ═══════════════════════════════════════════════════════════════════
print('[4/6] Signed volume...')
vol_cols_plot = [f'SIGNED_VOLUME_{i}' for i in range(2, 21)]
lags_v = list(range(2, 21))
 
fig = plt.figure(figsize=(18, 10), facecolor=BG)
fig.suptitle('FIG 4  ·  SIGNED VOLUME  (lag 1 dropped — 73% missing)',
             fontsize=13, fontweight='bold', color=ACCENT1, y=0.99)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)
 
# 4a. Boxplot per lag
ax = fig.add_subplot(gs[0, :2])
vol_data = [train[c].dropna() for c in vol_cols_plot]
bp = ax.boxplot(vol_data, patch_artist=True, showfliers=False,
                medianprops=dict(color=ACCENT2, linewidth=1.5),
                whiskerprops=dict(color=MUTED),
                capprops=dict(color=MUTED),
                boxprops=dict(linewidth=0.8))
for patch in bp['boxes']:
    patch.set_facecolor(ACCENT3)
    patch.set_alpha(0.5)
    patch.set_edgecolor(MUTED)
ax.set_xticks(range(1, len(vol_cols_plot) + 1))
ax.set_xticklabels([str(i) for i in lags_v], fontsize=7)
ax.set_xlabel('Lag  (2 = most recent usable)', fontsize=8)
ax.set_ylabel('Signed Volume', fontsize=8)
ax.set_title('SIGNED VOLUME DISTRIBUTION PER LAG  (outliers hidden)', fontsize=9, color=SUBTEXT, pad=6)
ax.axhline(0, color=TEXT, linewidth=0.6, linestyle='--', alpha=0.4)
ax.grid(axis='y', alpha=0.2)
 
# 4b. % positive by lag
ax = fig.add_subplot(gs[0, 2])
pct_pos = [(train[c] > 0).mean() * 100 for c in vol_cols_plot]
ax.plot(lags_v, pct_pos, color=ACCENT3, linewidth=1.5, marker='o',
        markersize=4, markerfacecolor=ACCENT1)
ax.axhline(50, color=TEXT, linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Lag', fontsize=8)
ax.set_ylabel('% Positive', fontsize=8)
ax.set_title('% POSITIVE SIGNED VOLUME BY LAG', fontsize=9, color=SUBTEXT, pad=6)
ax.set_ylim(45, 55)
ax.grid(alpha=0.2)
 
# 4c. Correlation with TARGET
ax = fig.add_subplot(gs[1, :2])
corrs_v = [train[c].corr(train['TARGET']) for c in vol_cols_plot]
bar_colors_v = [ACCENT3 if c > 0 else ACCENT2 for c in corrs_v]
ax.bar(lags_v, corrs_v, color=bar_colors_v, edgecolor='none')
ax.axhline(0, color=TEXT, linewidth=0.6, linestyle='--', alpha=0.5)
ax.set_xticks(lags_v)
ax.set_xticklabels([str(i) for i in lags_v], fontsize=7)
ax.set_xlabel('Lag', fontsize=8)
ax.set_ylabel('Pearson correlation', fontsize=8)
ax.set_title('CORRELATION: SIGNED_VOLUME_i  ↔  TARGET', fontsize=9, color=SUBTEXT, pad=6)
ax.grid(axis='y', alpha=0.2)
 
# 4d. SV_2 split by label
ax = fig.add_subplot(gs[1, 2])
for label, col, name in [(1, ACCENT1, 'POS (1)'), (0, ACCENT2, 'NEG (0)')]:
    sub = train[train['LABEL'] == label]['SIGNED_VOLUME_2']
    sub = sub.clip(sub.quantile(0.01), sub.quantile(0.99))
    ax.hist(sub, bins=80, alpha=0.6, color=col, label=name,
            edgecolor='none', density=True)
ax.set_xlabel('SIGNED_VOLUME_2', fontsize=8)
ax.set_ylabel('Density', fontsize=8)
ax.set_title('SV_2 SPLIT BY TARGET LABEL', fontsize=9, color=SUBTEXT, pad=6)
ax.legend(fontsize=7, framealpha=0.2)
ax.grid(alpha=0.2)
 
save(fig, 'fig4_signed_volume')
 
 


[4/6] Signed volume...
  ✓  outputs/fig4_signed_volume.png


In [20]:
# ═══════════════════════════════════════════════════════════════════
# FIG 5 — TURNOVER & GROUP
# ═══════════════════════════════════════════════════════════════════
print('[5/6] Turnover & group...')
fig = plt.figure(figsize=(18, 9), facecolor=BG)
fig.suptitle('FIG 5  ·  TURNOVER & GROUP ANALYSIS',
             fontsize=13, fontweight='bold', color=ACCENT1, y=0.99)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)
 
# 5a. Turnover distribution (log scale)
ax = fig.add_subplot(gs[0, 0])
ax.hist(np.log1p(train['MEDIAN_DAILY_TURNOVER'].dropna()),
        bins=100, color=ACCENT4, alpha=0.8, edgecolor='none')
ax.set_xlabel('log(1 + MDT)', fontsize=8)
ax.set_ylabel('Count', fontsize=8)
ax.set_title('TURNOVER DISTRIBUTION  (log scale)', fontsize=9, color=SUBTEXT, pad=6)
ax.grid(axis='y', alpha=0.2)
 
# 5b. Turnover by group
ax = fig.add_subplot(gs[0, 1])
for g, col in GROUP_COLORS.items():
    sub = np.log1p(train[train['GROUP'] == g]['MEDIAN_DAILY_TURNOVER'].dropna())
    ax.hist(sub, bins=80, alpha=0.5, color=col, label=f'G{g}',
            edgecolor='none', density=True)
ax.set_xlabel('log(1 + MDT)', fontsize=8)
ax.set_ylabel('Density', fontsize=8)
ax.set_title('TURNOVER BY GROUP', fontsize=9, color=SUBTEXT, pad=6)
ax.legend(fontsize=7, framealpha=0.2)
ax.grid(alpha=0.2)
 
# 5c. Turnover boxplot by group
ax = fig.add_subplot(gs[0, 2])
data_by_group = [np.log1p(train[train['GROUP'] == g]['MEDIAN_DAILY_TURNOVER'].dropna())
                 for g in sorted(GROUP_COLORS)]
bp = ax.boxplot(data_by_group, patch_artist=True, showfliers=False,
                medianprops=dict(color=BG, linewidth=2),
                whiskerprops=dict(color=MUTED),
                capprops=dict(color=MUTED))
for patch, col in zip(bp['boxes'], GROUP_COLORS.values()):
    patch.set_facecolor(col); patch.set_alpha(0.7); patch.set_edgecolor(MUTED)
ax.set_xticklabels([f'G{g}' for g in sorted(GROUP_COLORS)], fontsize=8)
ax.set_ylabel('log(1 + MDT)', fontsize=8)
ax.set_title('TURNOVER BOXPLOT BY GROUP', fontsize=9, color=SUBTEXT, pad=6)
ax.grid(axis='y', alpha=0.2)
 
# 5d. Intra-window return vol by group
ax = fig.add_subplot(gs[1, 0])
for g, col in GROUP_COLORS.items():
    sub = train[train['GROUP'] == g]['ret_vol']
    sub = sub.clip(0, sub.quantile(0.99))
    ax.hist(sub, bins=80, alpha=0.5, color=col, label=f'G{g}',
            edgecolor='none', density=True)
ax.set_xlabel('Std(RET_1…RET_20)', fontsize=8)
ax.set_ylabel('Density', fontsize=8)
ax.set_title('INTRA-WINDOW RETURN VOL BY GROUP', fontsize=9, color=SUBTEXT, pad=6)
ax.legend(fontsize=7, framealpha=0.2)
ax.grid(alpha=0.2)
 
# 5e. Mean window return by group
ax = fig.add_subplot(gs[1, 1])
for g, col in GROUP_COLORS.items():
    sub = train[train['GROUP'] == g]['ret_mean']
    sub = sub.clip(sub.quantile(0.01), sub.quantile(0.99))
    ax.hist(sub, bins=80, alpha=0.5, color=col, label=f'G{g}',
            edgecolor='none', density=True)
ax.axvline(0, color=TEXT, linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Mean(RET_1…RET_20)', fontsize=8)
ax.set_ylabel('Density', fontsize=8)
ax.set_title('MEAN WINDOW RETURN BY GROUP', fontsize=9, color=SUBTEXT, pad=6)
ax.legend(fontsize=7, framealpha=0.2)
ax.grid(alpha=0.2)
 
# 5f. Group distribution in TEST
ax = fig.add_subplot(gs[1, 2])
gc_test = X_test['GROUP'].value_counts().sort_index()
bars = ax.bar([f'G{g}' for g in gc_test.index], gc_test.values,
              color=[GROUP_COLORS.get(g, MUTED) for g in gc_test.index],
              edgecolor='none')
ax.set_title('GROUP DISTRIBUTION  (TEST)', fontsize=9, color=SUBTEXT, pad=6)
ax.set_ylabel('Count', fontsize=8)
ax.grid(axis='y', alpha=0.2)
for bar, val in zip(bars, gc_test.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val:,}', ha='center', va='bottom', fontsize=7, color=TEXT)
 
save(fig, 'fig5_turnover_group')
 
 


[5/6] Turnover & group...
  ✓  outputs/fig5_turnover_group.png


In [21]:
# ═══════════════════════════════════════════════════════════════════
# FIG 6 — CORRELATION HEATMAP
# ═══════════════════════════════════════════════════════════════════
print('[6/6] Correlation heatmap...')
fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor=BG)
fig.suptitle('FIG 6  ·  CROSS-FEATURE CORRELATION  (key features)',
             fontsize=13, fontweight='bold', color=ACCENT1, y=1.01)
 
cmap = sns.diverging_palette(220, 20, as_cmap=True)
key_cols = (
    [f'RET_{i}' for i in [1, 2, 3, 5, 10, 20]] +
    [f'SIGNED_VOLUME_{i}' for i in [2, 5, 10]] +
    ['MEDIAN_DAILY_TURNOVER', 'ret_vol', 'ret_mean', 'TARGET']
)
key_cols = [c for c in key_cols if c in train.columns]
 
for ax, (group_filter, title) in zip(axes, [(None, 'ALL GROUPS'), (1, 'GROUP 1')]):
    subset = train if group_filter is None else train[train['GROUP'] == group_filter]
    sample = subset.sample(min(50_000, len(subset)), random_state=42)
    corr = sample[key_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=ax, mask=mask, cmap=cmap, center=0,
                vmin=-0.15, vmax=0.15,
                annot=True, fmt='.2f', annot_kws={'size': 6},
                linewidths=0.3, linecolor=BG,
                cbar_kws={'shrink': 0.7})
    ax.set_title(title, fontsize=10, color=TEXT, pad=8)
    ax.tick_params(labelsize=7)
    ax.set_facecolor(PANEL)
 
plt.tight_layout()
save(fig, 'fig6_correlation_matrix')

[6/6] Correlation heatmap...
  ✓  outputs/fig6_correlation_matrix.png


In [22]:
# ═══════════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════════
print('\n── CLEANUP SUMMARY ───────────────────────────────────────────')
print('  DROPPED  : SIGNED_VOLUME_1  (73% missing in train, unusable)')
print('  FILLED   : RET_{i} NaNs → 0  (boundary artifacts, max 58 rows)')
print('  FILLED   : SIGNED_VOLUME_{i} NaNs → 0  (boundary + rollup gaps)')
print(f'  FILLED   : MEDIAN_DAILY_TURNOVER NaNs → {mdt_median:.6f}  (median)')
print(f'\n  Remaining missing in train: {train[key_cols].isnull().sum().sum()}')
print(f'  Output PNGs → {OUTPUT_DIR}/')
print('──────────────────────────────────────────────────────────────\n')
 


── CLEANUP SUMMARY ───────────────────────────────────────────
  DROPPED  : SIGNED_VOLUME_1  (73% missing in train, unusable)
  FILLED   : RET_{i} NaNs → 0  (boundary artifacts, max 58 rows)
  FILLED   : SIGNED_VOLUME_{i} NaNs → 0  (boundary + rollup gaps)
  FILLED   : MEDIAN_DAILY_TURNOVER NaNs → 0.015935  (median)

  Remaining missing in train: 0
  Output PNGs → outputs/
──────────────────────────────────────────────────────────────

